In [1]:
from omegaconf import DictConfig
import torch
import random

import numpy as np
import hydra
from hydra.utils import instantiate
from agents.factory import make_agent
from algorithms.ppo import PPO
from buffers.factory import make_buffer
from configs import register_configs
from configs.train import TrainConfig
from envs.factory import make_env
from envs.gym_env import GymEnv

import os

from runners.factory import make_runner
from trainers.base_trainer import BaseTrainer


lr_actor = 0.00003           # Learning rate
lr_critic = 0.0001

def set_global_seed(seed: int):
    SEED = seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@hydra.main(version_base=None, config_path="../config", config_name="train")
def main(cfg: DictConfig):

    config: TrainConfig = hydra.utils.instantiate(cfg)

    set_global_seed(config.seed)
    env = make_env(config.env)
    buffer = make_buffer(env.spec.obs_shape, env.spec.action_shape, config.buffer)
    agent = make_agent(env.spec.obs_shape, env.spec.action_shape, config.agent)
    runner = make_runner(env, agent, buffer, config.runner)
    optimizer = torch.optim.Adam([
        {'params': agent.architecture.actor_extractor.parameters(), 'lr': lr_actor},
        {'params': agent.architecture.actor_head.parameters(), 'lr': lr_actor},
        {'params': agent.architecture.critic_extractor.parameters(), 'lr': lr_critic},
        {'params': agent.architecture.critic_head.parameters(), 'lr': lr_critic}
    ])
    algorithm = PPO(optimizer)
    trainer = BaseTrainer(runner, algorithm)
    
    trainer.train()

if __name__ == "__main__":
    register_configs()
    main()

ModuleNotFoundError: No module named 'omegaconf'

In [3]:
!pip install hydra torch

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
    --------------------------------------- 2.4/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - -------------------------------------- 3.1/110.9 MB 11.6 MB/s eta 0:00:10
   - --------------------------------

  DEPRECATION: Building 'hydra' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'hydra'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  python setup.py bdist_wheel did not run successfully.
  exit code: 1
  
  [11 lines of output]
  C:\Users\dawid\anaconda3\Lib\site-packages\setuptools\_distutils\dist.py:268: UserWarning: Unknown distribution option: 'test_suite'
    warnings.warn(msg)
  running bdist_wheel
  running build
  running build_py
  creating build
  creating build\lib.win-amd64-cpython-313
  copying src\hydra.py -> build\lib.win-amd64-cpython-313
  running build_ext
  building '_hydra' extension
  error: Microsoft Visual